# Lab 07-08 · Filters and Resonance

Companion to chapter [07-08](../../chapters/07-08-filters-and-resonance.md). Handout: [README.md](README.md).

Responses are computed by solving each circuit's equations numerically at each frequency, and transients by integrating the state equations, so the chapter's formulas for $Q$, bandwidth and the Butterworth and Sallen–Key designs are being checked, not assumed. Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import math
import shutil
import subprocess
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq, minimize_scalar
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)
dB = lambda z: 20 * np.log10(np.abs(z))


def ngspice_values(path):
    out = subprocess.run(["ngspice", "-b", path], capture_output=True, text=True).stdout
    vals = {}
    for line in out.splitlines():
        parts = line.split("=")
        if len(parts) >= 2:
            try:
                vals[parts[0].strip()] = float(parts[1].split()[0])
            except (ValueError, IndexError):
                pass
    return vals

ngspice available: True


## Part 1 — One Q, four ways

Example 07-08.1: $R$ = 10 Ω, $L$ = 100 µH, $C$ = 100 pF.

**🔮 Predict:** after a kick, how many cycles does the oscillation take to decay by a factor $e$?

In [2]:
R, L, C = 10.0, 100e-6, 100e-12
w0 = 1 / math.sqrt(L * C); f0 = w0 / (2 * math.pi)
I = lambda f: 1 / (R + 1j * (2 * math.pi * f * L - 1 / (2 * math.pi * f * C)))   # current per volt
Ipk = abs(I(f0))
f1 = brentq(lambda f: abs(I(f)) - Ipk / math.sqrt(2), 1.5e6, f0)
f2 = brentq(lambda f: abs(I(f)) - Ipk / math.sqrt(2), f0, 1.7e6)
Q_bw = f0 / (f2 - f1)
# energy: stored and lost per cycle at resonance, from sampled waveforms
t = np.linspace(0, 1 / f0, 20001); i = np.cos(w0 * t); vC = np.sin(w0 * t) / (w0 * C)
W = np.mean(0.5 * L * i**2 + 0.5 * C * vC**2); E = np.trapezoid(i**2 * R, t)
Q_energy = 2 * math.pi * W / E
# pole location
p = np.roots([L * C, R * C, 1])[0]
Q_pole = abs(p) / (-2 * p.real)
# ring-down: free response of the series circuit, state (i, vC)
A = np.array([[-R / L, -1 / L], [1 / C, 0]])
sol = solve_ivp(lambda t, x: A @ x, (0, 60 / f0), [1e-3, 0], max_step=1 / (f0 * 100), rtol=1e-10, atol=1e-15, dense_output=True)
tt = np.linspace(0, 60 / f0, 60 * 400); ii = sol.sol(tt)[0]
peaks = tt[1:-1][(ii[1:-1] > ii[:-2]) & (ii[1:-1] > ii[2:])]
amps = np.interp(peaks, tt, ii)
tau_env = -1 / np.polyfit(peaks, np.log(amps), 1)[0]
Q_ring = math.pi * f0 * tau_env
print(f"Q from bandwidth {Q_bw:.3f}, energy {Q_energy:.3f}, poles {Q_pole:.3f}, ring-down {Q_ring:.3f}")
print(f"cycles to decay by e: {tau_env * f0:.2f}")
assert all(abs(q - 100) < 0.05 for q in (Q_bw, Q_energy, Q_pole, Q_ring))
if HAVE_NGSPICE:
    print("ngspice:", ngspice_values("ex1_series.cir"))

Q from bandwidth 100.000, energy 100.000, poles 100.000, ring-down 100.003
cycles to decay by e: 31.83
ngspice: {'Doing analysis at TEMP': 27.0, 'vrpk': 0.01, 'f0': 1591550.0, 'ipk': 0.001, 'f1': 1583612.0, 'f2': 1599527.0, 'vc0': 1.0}


## Part 2 — Loading a tank

Example 07-08.2: a 1 µH coil with $Q_c$ = 80 at 10.7 MHz, driven by 20 kΩ and loaded by $R_{load}$.

**🔮 Predict:** which resistor takes the largest share of the power? What happens to the bandwidth as the load rises to 100 kΩ?

In [3]:
f0t, Lc, Qc = 10.7e6, 1e-6, 80
w0t = 2 * math.pi * f0t; rc = w0t * Lc / Qc
Yc = 1 / (rc + 1j * w0t * Lc); Ct = -Yc.imag / w0t


def tank(f, Rload):
    w = 2 * math.pi * f
    return 1 / (1 / (rc + 1j * w * Lc) + 1j * w * Ct + 1 / 20e3 + 1 / Rload)


for Rload in (10e3, 30e3, 100e3, 1e9):
    zpk = abs(tank(f0t, Rload))
    a = brentq(lambda f: abs(tank(f, Rload)) - zpk / math.sqrt(2), 9e6, f0t)
    b = brentq(lambda f: abs(tank(f, Rload)) - zpk / math.sqrt(2), f0t, 12e6)
    V = tank(f0t, Rload)
    Pc = 0.5 * abs(V / (rc + 1j * w0t * Lc)) ** 2 * rc; Ps = 0.5 * abs(V) ** 2 / 20e3; Pl = 0.5 * abs(V) ** 2 / Rload
    tot = Pc + Ps + Pl
    print(f"load {Rload:9.3g}: bandwidth {(b - a) / 1e3:6.1f} kHz; coil {Pc / tot:5.1%}, source {Ps / tot:5.1%}, load {Pl / tot:5.1%}")

load     1e+04: bandwidth  241.7 kHz; coil 55.3%, source 14.9%, load 29.8%
load     3e+04: bandwidth  193.7 kHz; coil 69.0%, source 18.6%, load 12.4%
load     1e+05: bandwidth  176.9 kHz; coil 75.6%, source 20.3%, load  4.1%
load     1e+09: bandwidth  169.8 kHz; coil 78.8%, source 21.2%, load  0.0%


## Part 3 — Butterworth sections

**🔮 Predict:** the $Q$ of the highest-$Q$ section of a sixth-order Butterworth filter.

In [4]:
def butterworth_sections(n):
    roots = np.roots(np.r_[(-1) ** n, np.zeros(2 * n - 1), 1])
    lhp = roots[roots.real < -1e-9]
    qs = sorted({float(round(abs(r) / (-2 * r.real), 6)) for r in lhp if abs(r.imag) > 1e-9})
    return qs, int(np.sum(np.abs(lhp.imag) < 1e-9))


for n in range(1, 9):
    qs, nreal = butterworth_sections(n)
    print(f"order {n}: {nreal} real pole(s); pair Q values {qs}")
assert abs(max(butterworth_sections(6)[0]) - 1.932) < 0.001

order 1: 1 real pole(s); pair Q values []
order 2: 0 real pole(s); pair Q values [0.707107]
order 3: 1 real pole(s); pair Q values [1.0]
order 4: 0 real pole(s); pair Q values [0.541196, 1.306563]
order 5: 1 real pole(s); pair Q values [0.618034, 1.618034]
order 6: 0 real pole(s); pair Q values [0.517638, 0.707107, 1.931852]
order 7: 1 real pole(s); pair Q values [0.554958, 0.801938, 2.24698]
order 8: 0 real pole(s); pair Q values [0.509796, 0.601345, 0.899976, 2.562915]


## Part 4 — Tolerances and the op-amp

Example 07-08.5's unity-gain Sallen–Key section (11.3 kΩ, 20 nF, 10 nF). First a Monte Carlo over capacitor tolerance (uniformly distributed within ±tol), then the guided exercise's finite gain–bandwidth product.

**🔮 Predict:** with 5% capacitors, what spread of cutoff frequency do you expect?

In [5]:
rng = np.random.default_rng(7)


def sk(f, R1, R2, C1, C2, gbw=None, A0=1e5):
    s = 2j * math.pi * f
    K = 1 if gbw is None else (A0 / (1 + s * A0 / (2 * math.pi * gbw))) / (1 + A0 / (1 + s * A0 / (2 * math.pi * gbw)))
    return K / (1 + s * (R1 * C2 + R2 * C2 + (1 - K) * R1 * C1) + s * s * R1 * R2 * C1 * C2)


for tol in (0.05, 0.01):
    C1s = 20e-9 * (1 + rng.uniform(-tol, tol, 5000)); C2s = 10e-9 * (1 + rng.uniform(-tol, tol, 5000))
    f0s = 1 / (2 * math.pi * 11.3e3 * np.sqrt(C1s * C2s)); Qs = 0.5 * np.sqrt(C1s / C2s)
    print(f"tol {tol:.0%}: f0 = {f0s.mean():.1f} Hz +/- {f0s.std():.1f} (1 sigma), Q from {Qs.min():.3f} to {Qs.max():.3f}")
for gbw in (None, 1e6, 100e3, 10e3):
    print(f"GBW {gbw}: gain at 1 kHz {dB(sk(1e3, 11.3e3, 11.3e3, 20e-9, 10e-9, gbw)):.3f} dB")
assert abs(dB(sk(1e3, 11.3e3, 11.3e3, 20e-9, 10e-9, 100e3)) + 3.048) < 0.001
if HAVE_NGSPICE:
    print("ngspice:", ngspice_values("ex5_sallen_key.cir"))

tol 5%: f0 = 996.5 Hz +/- 20.7 (1 sigma), Q from 0.673 to 0.742
tol 1%: f0 = 995.9 Hz +/- 4.1 (1 sigma), Q from 0.700 to 0.714
GBW None: gain at 1 kHz -3.046 dB
GBW 1000000.0: gain at 1 kHz -3.046 dB
GBW 100000.0: gain at 1 kHz -3.048 dB
GBW 10000.0: gain at 1 kHz -3.221 dB


ngspice: {'Doing analysis at TEMP': 27.0, 'f0_1': 995.9251, 'f0_2': 1049.797, 'q_2': 0.6708191, 'g1k_1': -3.04593, 'g1k_3': -3.04832, 'g1k_4': -3.22123}


## Part 5 — Insertion loss and resonator Q

One parallel resonator between equal terminations: loss $= -20\lg(1 - Q_L/Q_u)$.

**🔮 Predict:** the unloaded $Q$ needed for 0.5 dB of loss when $Q_L$ = 100.

In [6]:
QL = 100
Qu = np.logspace(2.01, 4.5, 400)
loss = -20 * np.log10(1 - QL / Qu)
fig, ax = plt.subplots(figsize=(6, 3.5)); ax.loglog(Qu, loss); ax.set_xlabel("unloaded Q"); ax.set_ylabel("loss (dB)"); plt.close(fig)
Qu05 = brentq(lambda q: -20 * math.log10(1 - QL / q) - 0.5, 150, 1e5)
print(f"Q_u for 0.5 dB: {Qu05:.0f}")
assert abs(Qu05 - 1788) < 1

Q_u for 0.5 dB: 1788


## Your turn

1. A parallel tank has $L$ = 10 µH, $C$ = 1 nF and $R$ = 20 kΩ. Find $f_0$, $Q$ and the bandwidth.
2. An equal-component Sallen–Key low-pass must have $Q$ = 2. What gain $K$ does it need, and what is the $Q$ sensitivity $KQ$?
3. What order of Butterworth filter gives at least 60 dB at three times the cutoff?

<details><summary>Answers</summary>

1. $f_0 = 1/(2\pi\sqrt{LC})$ = **1.592 MHz**; $Z_0 = \sqrt{L/C}$ = 100 Ω, so $Q = R/Z_0$ = **200** and the bandwidth is **7.96 kHz**.
2. $K = 3 - 1/Q$ = **2.5**, and $KQ$ = **5**: a 1% gain error moves $Q$ by 5%.
3. $n \ge \lg(10^6 - 1)/(2\lg3)$ = 6.29, so **7**.

</details>

In [7]:
f0y = 1 / (2 * math.pi * math.sqrt(10e-6 * 1e-9)); Qy = 20e3 / math.sqrt(10e-6 / 1e-9)
print(f"1) f0 = {f0y / 1e6:.3f} MHz, Q = {Qy:.0f}, bandwidth = {f0y / Qy / 1e3:.2f} kHz")
print(f"2) K = {3 - 1 / 2}, KQ = {(3 - 1 / 2) * 2}")
print(f"3) n >= {math.log10(1e6 - 1) / (2 * math.log10(3)):.2f}")
assert abs(f0y / Qy - 7958) < 1 and math.ceil(math.log10(1e6 - 1) / (2 * math.log10(3))) == 7

1) f0 = 1.592 MHz, Q = 200, bandwidth = 7.96 kHz
2) K = 2.5, KQ = 5.0
3) n >= 6.29
